In [ ]:
import math, json
from pathlib import Path
import soundfile as sf
import numpy as np
import pandas as pd
import librosa
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

In [ ]:
import glob
import re

In [ ]:
import os, gc, torch
from transformers import AutoFeatureExtractor, Wav2Vec2Model

# splits del 15

In [ ]:
TEST_SIZE   = 0.15
RAND        = 42

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
UNIFIED = Path("/content/drive/MyDrive/TP1/data/unificado.csv")
OUT_DIR  = Path("/content/drive/MyDrive/TP1/data/deriv")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def split_per_lang(df, test_size=0.15, random_state=42):
    """
    Split por idioma manteniendo grupos por clip_id
    Retorna df_train, df_test.
    """
    parts = []
    for lang, d in df.groupby("lang", dropna=False):
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        idx_tr, idx_te = next(gss.split(d, groups=d["clip_id"]))
        parts.append(("train", d.iloc[idx_tr]))
        parts.append(("test",  d.iloc[idx_te]))
    df_tr = pd.concat([p for tag,p in parts if tag=="train"], ignore_index=True)
    df_te = pd.concat([p for tag,p in parts if tag=="test"],  ignore_index=True)
    return df_tr, df_te

In [ ]:
df = pd.read_csv(UNIFIED)
assert set(["path","corpus","lang","speaker_id","clip_id","valence","arousal","dominance"]).issubset(df.columns)

df_tr, df_te = split_per_lang(df, TEST_SIZE, RAND)

In [ ]:
# chequeo
print("Split por idioma:")
print(df.groupby("lang").size().rename("total"))
print("train:", df_tr.groupby("lang").size().rename("n"))
print("test:",  df_te.groupby("lang").size().rename("n"))

Split por idioma:
lang
es     3763
qu    12420
Name: total, dtype: int64
train: lang
es     3198
qu    10557
Name: n, dtype: int64
test: lang
es     565
qu    1863
Name: n, dtype: int64


In [ ]:
split_dir = OUT_DIR/"splits"
split_dir.mkdir(exist_ok=True, parents=True)
df_tr.to_csv(split_dir/"train_unified.csv", index=False)
df_te.to_csv(split_dir/"test_unified.csv",  index=False)
print("Guardados splits en", split_dir)

Guardados splits en /content/drive/MyDrive/TP1/data/deriv/splits


# data augmentation

In [ ]:
DERIV_DIR   = Path("/content/drive/MyDrive/TP1/data/deriv")
SPLIT_DIR   = DERIV_DIR / "splits"
AUG_DIR     = DERIV_DIR / "v3/train_aug" / "es"     # salida para WAV aumentados (solo español)

In [ ]:
SR          = 16000
np.random.seed(42)

In [ ]:
N_VARIANTS_PER_CLIP = 2      # solo clips ES si balanceás
P_STRETCH = 0.45
P_PITCH   = 0.1              # o 0.0 para apagar
P_NOISE   = 0.65
P_REVERB  = 0.25

STRETCH_RANGE = (0.985, 1.015)
PITCH_RANGE   = (-0.3, 0.3)
SNR_RANGE_DB  = (25.0, 35.0)
REVERB_RT60   = (0.10, 0.25)
REVERB_MIX    = (0.05, 0.12)

In [ ]:
AUG_DIR.mkdir(parents=True, exist_ok=True)

### funciones

In [ ]:
def add_noise_snr(y, snr_db):
    """Añade ruido blanco para alcanzar SNR objetivo (en dB)."""
    sig_rms = np.sqrt(np.mean(y**2) + 1e-12)
    if sig_rms < 1e-6:
        return y  # no inflar silencios
    noise = np.random.randn(len(y)).astype(np.float32)
    noise = noise / (np.sqrt(np.mean(noise**2)) + 1e-12)
    noise_rms = sig_rms / (10.0 ** (snr_db / 20.0))
    y_noisy = y + noise * noise_rms
    return y_noisy

In [ ]:
def time_stretch(y, rate):
    if rate <= 0:
        return y
    return librosa.effects.time_stretch(y, rate=rate)

In [ ]:
def pitch_shift(y, sr, n_steps):
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

In [ ]:
def add_reverb_simple(y, sr, rt60=0.25, mix=0.15):
    """Reverb muy ligera por convolución con impulso exponencial (rápido y simple)."""
    # Calcular tau tal que exp(-rt60/tau) ≈ 0.001  (−60 dB)
    tau = rt60 / np.log(1000.0)
    n = int(rt60 * sr)
    if n < 1:
        return y
    t = np.arange(n) / sr
    h = np.exp(-t / tau).astype(np.float32)
    h = h / np.sum(h)  # normaliza energía
    # Convolución
    y_rev = np.convolve(y, h, mode="full")[:len(y)]
    # Mezcla húmedo/seco
    out = (1.0 - mix) * y + mix * y_rev
    return out.astype(np.float32)

aleatoriedad para que cada clip tenga variaciones distintas y así el modelo aprenda a generalizar

In [ ]:
rng = np.random.default_rng(123)

In [ ]:
def augment_one(y, sr):
    ops = ["stretch", "pitch", "noise", "reverb"]
    w   = np.array([P_STRETCH, P_PITCH, P_NOISE, P_REVERB], dtype=float)
    if not np.isfinite(w).all() or w.sum() <= 0: w[:] = 1.0
    w /= w.sum()
    op = rng.choice(ops, p=w)

    if op == "stretch":
        rate = float(rng.uniform(*STRETCH_RANGE))
        y2 = librosa.effects.time_stretch(y, rate=rate)
        tag = f"ts{rate:.3f}"

    elif op == "pitch":
        steps = float(rng.uniform(*PITCH_RANGE))
        y2 = librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)
        tag = f"ps{steps:+.2f}"

    elif op == "noise":
        snr = float(rng.uniform(*SNR_RANGE_DB))
        y2  = add_noise_snr(y, snr)
        tag = f"ns{snr:.1f}"

    else:  # reverb
        rt60 = float(rng.uniform(*REVERB_RT60))
        mix  = float(rng.uniform(*REVERB_MIX))
        y2   = add_reverb_simple(y, sr, rt60=rt60, mix=mix)
        tag  = f"rvb{rt60:.2f}s_{mix:.2f}"

    # Limitador de seguridad (no normaliza, solo evita clip)
    y2 = np.clip(y2, -0.999, 0.999).astype(np.float32)
    return y2, tag

In [ ]:
df_tr = pd.read_csv(SPLIT_DIR / "train_unified.csv")

In [ ]:
df_es = df_tr[df_tr["lang"] == "es"].reset_index(drop=True)

In [ ]:
print("Clips train totales:", len(df_tr), " | Español:", len(df_es))

Clips train totales: 13755  | Español: 3198


## resultado

In [ ]:
rows_new = []

for i, row in tqdm(df_es.iterrows(), total=len(df_es), desc="Augment español"):
    in_path = Path(row["path"])
    # Carga audio normalizado (mono 16 kHz ya)
    y, sr = sf.read(str(in_path), always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr != SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR, res_type="soxr_vhq")
        sr = SR
    y = y.astype(np.float32)

    stem = in_path.stem
    for k in range(N_VARIANTS_PER_CLIP):
        y_aug, ops = augment_one(y, sr)
        out_name = f"{stem}_aug{k+1}_{ops}.wav"
        out_path = AUG_DIR / out_name
        sf.write(str(out_path), y_aug, sr, subtype="PCM_16")

        # fila nueva (copia + cambios mínimos)
        new_row = row.copy()
        new_row["path_norm"] = str(out_path)
        new_row["sr"] = sr
        new_row["is_aug"] = True
        new_row["aug_ops"] = ops
        rows_new.append(new_row)

df_aug = pd.DataFrame(rows_new)
print("Nuevos ejemplos creados:", len(df_aug))

Augment español:   0%|          | 0/3198 [00:00<?, ?it/s]

Nuevos ejemplos creados: 6396


In [ ]:
df_aug

,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance,path_norm,sr,is_aug,aug_ops
0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_10-02-03-04,2.0,3.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000,True,ns25.5
0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_10-02-03-04,2.0,3.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000,True,ts0.991
1,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_104-03-02-04,3.0,2.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000,True,ts1.009
1,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_104-03-02-04,3.0,2.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000,True,rvb0.14s_0.11
2,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_1-02-03-03,2.0,3.0,3.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000,True,rvb0.18s_0.07
...,...,...,...,...,...,...,...,...,...,...,...,...
3195,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_83-03-03-04,3.0,3.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000,True,ts1.013
3196,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_99-02-02-04,2.0,2.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000,True,ns32.2
3196,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_99-02-02-04,2.0,2.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000,True,ns33.2
3197,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_86-03-03-04,3.0,3.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000,True,rvb0.12s_0.11


In [ ]:
df_aug.shape

(6396, 12)

In [ ]:
df_tr_combined = pd.concat([df_tr, df_aug], ignore_index=True)
out_csv = DERIV_DIR / "v3/train_unified_aug.csv"
df_tr_combined.to_csv(out_csv, index=False)
print("Guardado:", out_csv)
print("Resumen por idioma:")
print(df_tr_combined["lang"].value_counts())
print("Cols nuevas disponibles:", [c for c in df_tr_combined.columns if c in ["is_aug","aug_ops","dbfs_norm","sr"]])


Guardado: /content/drive/MyDrive/TP1/data/deriv/v3/train_unified_aug.csv
Resumen por idioma:
lang
qu    10557
es     9594
Name: count, dtype: int64
Cols nuevas disponibles: ['sr', 'is_aug', 'aug_ops']


# ventaneo

In [ ]:
BASE = Path("/content/drive/MyDrive/TP1/data/deriv/v3")
OUT_DIR   = BASE / "windows"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = BASE / "train_unified_aug.csv"
TEST_CSV  = SPLIT_DIR / "test_unified.csv"

In [ ]:
SR  = 16000
WIN_SEC  = 1.0
HOP_SEC  = 0.5
MAX_WINDOWS_PER_CLIP = 10
MIN_WINDOWS_PER_CLIP = 1

### funciones

In [ ]:
def load_mono_sr16(path):
    """Lee WAV, fuerza mono y SR=16k si hace falta, float32 [-1,1]."""
    y, sr = sf.read(str(path), always_2d=False)
    # a mono
    if y.ndim == 2:
        y = y.mean(axis=1)
    # a float32
    y = y.astype(np.float32, copy=False)
    # remuestrea si hace falta
    if sr != SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR, res_type="soxr_vhq")
        sr = SR
    return y, sr

In [ ]:
def frame_indices(n_samples, win_s, hop_s, sr, ensure_min=True):
    """Devuelve arrays start/end en muestras para ventaneo deslizante.
       Si el audio es muy corto, crea 1 ventana centrada (reflect-pad lógico)."""
    win = int(round(win_s * sr))
    hop = int(round(hop_s * sr))
    if n_samples <= 0:
        return np.array([0], dtype=np.int64), np.array([0], dtype=np.int64)

    if n_samples < win:
        if not ensure_min:
            return np.array([], dtype=np.int64), np.array([], dtype=np.int64)
        # forzamos una ventana centrada
        center = n_samples // 2
        start = max(0, center - win // 2)
        end   = min(n_samples, start + win)
        start = max(0, end - win)  # corrige borde
        return np.array([start], dtype=np.int64), np.array([end], dtype=np.int64)

    starts = np.arange(0, n_samples - win + 1, hop, dtype=np.int64)
    ends   = starts + win
    return starts, ends

In [ ]:
def equispaced_idx(n, k):
    """Devuelve k índices aproximadamente equiespaciados en [0..n-1]."""
    if k >= n:
        return np.arange(n, dtype=np.int64)
    # usar linspace y redondear
    pos = np.linspace(0, n - 1, num=k)
    return np.unique(np.round(pos).astype(np.int64))

In [ ]:
def rms_vec(x):
    """RMS de un vector."""
    return float(np.sqrt(np.mean(x * x) + 1e-12))

In [ ]:
def windows_from_df(df, split_name="train"):
    """Genera ventanas y meta por cada clip del DataFrame."""
    win_clip_idx, win_start, win_end = [], [], []
    win_path, win_lang, win_is_aug, win_ops = [], [], [], []
    win_rms = []

    for i, row in tqdm(df.iterrows(), total=len(df), desc=f"Ventaneo {split_name}"):
        p = row.get("path_norm")
        if pd.isna(p):
            p = row.get("path")
            if pd.isna(p):
                continue

        y, sr = load_mono_sr16(p)
        s_arr, e_arr = frame_indices(len(y), WIN_SEC, HOP_SEC, sr=sr, ensure_min=True)

        # limitar número de ventanas por clip
        if len(s_arr) > MAX_WINDOWS_PER_CLIP:
            keep = equispaced_idx(len(s_arr), MAX_WINDOWS_PER_CLIP)
            s_arr, e_arr = s_arr[keep], e_arr[keep]

        # asegurar mínimo
        if len(s_arr) < MIN_WINDOWS_PER_CLIP:
            # ya garantizado por frame_indices, pero por si acaso
            s_arr = np.array([0], dtype=np.int64)
            e_arr = np.array([min(len(y), int(WIN_SEC*sr))], dtype=np.int64)


        # RMS por ventana
        for s, e in zip(s_arr, e_arr):
            seg = y[s:e]
            win_clip_idx.append(i)
            win_start.append(int(s))
            win_end.append(int(e))
            win_path.append(p)
            win_lang.append(row.get("lang", "xx"))
            is_aug_val = row["is_aug"] if "is_aug" in row else False
            if pd.isna(is_aug_val):
                is_aug_val = False
            win_is_aug.append(bool(is_aug_val))

            ops_val = row["aug_ops"] if "aug_ops" in row else ""
            if pd.isna(ops_val):
                ops_val = ""
            win_ops.append(str(ops_val))
            win_rms.append(rms_vec(seg))

    # a DataFrame
    dfw = pd.DataFrame({
        "clip_idx": win_clip_idx,
        "path": win_path,
        "lang": win_lang,
        "is_aug": win_is_aug,
        "aug_ops": win_ops,
        "start_samp": win_start,
        "end_samp": win_end,
        "win_rms": win_rms,
    })
    return dfw

## resultado

In [ ]:
df_tr = pd.read_csv(TRAIN_CSV)
df_te = pd.read_csv(TEST_CSV)

In [ ]:
for df in (df_tr, df_te):
    # is_aug: booleano (nullable) -> rellena False
    if "is_aug" in df.columns:
        df["is_aug"] = df["is_aug"].astype("boolean").fillna(False)
    else:
        df["is_aug"] = pd.Series(False, index=df.index, dtype="boolean")

    # aug_ops: texto -> rellena ""
    if "aug_ops" in df.columns:
        df["aug_ops"] = df["aug_ops"].astype("string").fillna("")
    else:
        df["aug_ops"] = pd.Series("", index=df.index, dtype="string")

In [ ]:
for name, df in [("train", df_tr), ("test", df_te)]:
    assert "path" in df.columns, f"Falta path_norm en {name}"
    if "lang" not in df.columns:
        df["lang"] = "xx"

In [ ]:
df_tr

,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance,path_norm,sr,is_aug,aug_ops
0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_10-02-03-04,2.0,3.0,4.0,NaN,NaN,False,
1,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_104-03-02-04,3.0,2.0,4.0,NaN,NaN,False,
2,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_1-02-03-03,2.0,3.0,3.0,NaN,NaN,False,
3,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_105-03-02-04,3.0,2.0,4.0,NaN,NaN,False,
4,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_102-03-04-05,3.0,4.0,5.0,NaN,NaN,False,
...,...,...,...,...,...,...,...,...,...,...,...,...
20146,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_83-03-03-04,3.0,3.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000.0,True,ts1.013
20147,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_99-02-02-04,2.0,2.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000.0,True,ns32.2
20148,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_99-02-02-04,2.0,2.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000.0,True,ns33.2
20149,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_86-03-03-04,3.0,3.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000.0,True,rvb0.12s_0.11


In [ ]:
# fill path_norm Nan with path
df_tr = df_tr.fillna({"path_norm": df_tr["path"]})

In [ ]:
df_tr = df_tr.fillna({"is_aug": 'False'})

In [ ]:
df_tr= df_tr.fillna({"sr": ''})
df_tr= df_tr.fillna({"aug_ops": ''})

In [ ]:
df_tr.shape

(20151, 12)

In [ ]:
dfw_tr = windows_from_df(df_tr, "train")

Ventaneo train:   0%|          | 0/20151 [00:00<?, ?it/s]

In [ ]:
dfw_tr.shape

(132621, 8)

In [ ]:
dfw_tr.to_csv(OUT_DIR / "pre-windows_train_v3.csv", index=False)

In [ ]:
df_te

,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance,is_aug,aug_ops
0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_100-03-03-04,3.00,3.00,4.00,False,
1,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_101-03-04-04,3.00,4.00,4.00,False,
2,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_103-03-03-04,3.00,3.00,4.00,False,
3,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_128-03-02-05,3.00,2.00,5.00,False,
4,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_113-03-02-02,3.00,2.00,2.00,False,
...,...,...,...,...,...,...,...,...,...,...
2423,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,10397,10397-4.25-2.75-3.5,4.25,2.75,3.50,False,
2424,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,10399,10399-3-2.75-3,3.00,2.75,3.00,False,
2425,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,10400,10400-3.25-2.25-2,3.25,2.25,2.00,False,
2426,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,10409,10409-3.5-2.25-2.5,3.50,2.25,2.50,False,


In [ ]:
dfw_te = windows_from_df(df_te, "test")

Ventaneo test:   0%|          | 0/2428 [00:00<?, ?it/s]

In [ ]:
dfw_te.shape

(15732, 8)

In [ ]:
dfw_te.to_csv(OUT_DIR / "pre-windows_test_v3.csv", index=False)

In [ ]:
# también guardo como CSV para inspección
dfw_tr.to_csv(OUT_DIR / "pre-windows_train_v3.csv", index=False)
dfw_te.to_csv(OUT_DIR / "pre-windows_test_v3.csv", index=False)

In [ ]:
dfw_tr = pd.read_csv(OUT_DIR / "pre-windows_train_v3.csv")

/tmp/ipython-input-1831370360.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  dfw_tr = pd.read_csv(OUT_DIR / "pre-windows_train_v3.csv")


In [ ]:
# count nan values in aug_ops
dfw_tr["aug_ops"].isna().sum()

np.int64(88626)

In [ ]:
dfw_tr['aug_ops'].fillna('', inplace=True)

In [ ]:
dfw_te = pd.read_csv(OUT_DIR / "pre-windows_test_v3.csv")

In [ ]:
dfw_te["aug_ops"].isna().sum()

np.int64(15732)

In [ ]:
dfw_te['aug_ops'].fillna('', inplace=True)

In [ ]:
print("Ventanas train:", len(dfw_tr), " | Ventanas test:", len(dfw_te))
print(dfw_tr.head(3))

Ventanas train: 132621  | Ventanas test: 15732
   clip_idx                                               path lang  is_aug  \
0         0  /content/drive/.shortcut-targets-by-id/1Kj75Ss...   es   False   
1         0  /content/drive/.shortcut-targets-by-id/1Kj75Ss...   es   False   
2         0  /content/drive/.shortcut-targets-by-id/1Kj75Ss...   es   False   

  aug_ops  start_samp  end_samp   win_rms  
0                   0     16000  0.177848  
1                8000     24000  0.182647  
2               24000     40000  0.201038  


In [ ]:
mu_rms = dfw_tr["win_rms"].mean()
sd_rms = dfw_tr["win_rms"].std(ddof=1) + 1e-9

dfw_tr["zrms"] = (dfw_tr["win_rms"] - mu_rms) / sd_rms
dfw_te["zrms"] = (dfw_te["win_rms"] - mu_rms) / sd_rms

dfw_tr["zrms"] = np.clip(dfw_tr["zrms"], -6, 6)
dfw_te["zrms"] = np.clip(dfw_te["zrms"], -6, 6)

print("RMS μ, σ (train):", mu_rms, sd_rms)

RMS μ, σ (train): 0.11410142223244359 0.10235130688224367


In [ ]:
eps = 1e-6
dfw_tr["zrms_log"] = (np.log1p(dfw_tr["win_rms"]+eps) - np.log1p(mu_rms+eps)) / (np.std(np.log1p(dfw_tr["win_rms"]+eps), ddof=1) + 1e-9)
dfw_te["zrms_log"] = (np.log1p(dfw_te["win_rms"]+eps) - np.log1p(mu_rms+eps)) / (np.std(np.log1p(dfw_tr["win_rms"]+eps), ddof=1) + 1e-9)


In [ ]:
def to_npz(dfw, out_path):
    np.savez_compressed(
        out_path,
        path=np.array(dfw["path"].tolist()),
        clip_idx=dfw["clip_idx"].to_numpy(np.int64),
        start_samp=dfw["start_samp"].to_numpy(np.int64),
        end_samp=dfw["end_samp"].to_numpy(np.int64),
        win_rms=dfw["win_rms"].to_numpy(np.float32),
        zrms=dfw["zrms"].to_numpy(np.float32),
        lang=np.array(dfw["lang"].tolist()),
        is_aug=dfw["is_aug"].to_numpy(bool),
        aug_ops=np.array(dfw["aug_ops"].astype(str).tolist()),
        sr=np.int64(SR),
        win_sec=np.float32(WIN_SEC),
        hop_sec=np.float32(HOP_SEC),
        max_win_per_clip=np.int64(MAX_WINDOWS_PER_CLIP),
        min_win_per_clip=np.int64(MIN_WINDOWS_PER_CLIP),
        mu_rms=np.float32(mu_rms),
        sd_rms=np.float32(sd_rms),
    )

to_npz(dfw_tr, OUT_DIR / "windows_train_v3.npz")
to_npz(dfw_te, OUT_DIR / "windows_test_v3.npz")

# también guardo como CSV para inspección
dfw_tr.to_csv(OUT_DIR / "windows_train_v3.csv", index=False)
dfw_te.to_csv(OUT_DIR / "windows_test_v3.csv", index=False)

print("Guardado:\n", OUT_DIR / "windows_train_v3.npz", "\n", OUT_DIR / "windows_test_v3.npz")

Guardado:
 /content/drive/MyDrive/TP1/data/deriv/v3/windows/windows_train_v3.npz 
 /content/drive/MyDrive/TP1/data/deriv/v3/windows/windows_test_v3.npz


## nuevas agregaciones

In [ ]:
!pip install torch torchaudio torchcrepe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.3 MB/s eta 0:00:00


In [ ]:
import torch
import torchaudio
import torchcrepe
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

In [ ]:
BASE_V3 = Path("/content/drive/MyDrive/TP1/data/deriv/v3")
WIN_DIR = BASE_V3 / "windows"
TRAIN_WIN_NPZ = WIN_DIR / "windows_train_v3.npz"
TEST_WIN_NPZ = WIN_DIR / "windows_test_v3.npz"

# salida para las nuevas características
ACOUSTIC_FEAT_DIR = BASE_V3 / "acoustic_features"
ACOUSTIC_FEAT_DIR.mkdir(parents=True, exist_ok=True)

SR = 16000

In [ ]:
SHARD_OUTPUT_DIR = Path("/content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features_shards_train")

In [ ]:
MIN_RMS_THRESHOLD = 0.001
FILES_PER_SHARD = 1000

In [ ]:
def extract_features(npz_path: Path, out_dir: Path):
    """
    Extrae features, guarda progresivamente y puede reanudar si se interrumpe.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Usando dispositivo: {device}")

    out_dir.mkdir(parents=True, exist_ok=True)

    mfcc_transform = torchaudio.transforms.MFCC(
        sample_rate=SR, n_mfcc=13,
        melkwargs={"n_fft": 400, "hop_length": 160, "n_mels": 80},
    ).to(device)
    HOP_LENGTH_MFCC = 160

    D = np.load(npz_path, allow_pickle=True)
    df_win = pd.DataFrame({"path": D["path"], "start_samp": D["start_samp"], "end_samp": D["end_samp"]})

    # Agrupamos los paths para poder saltar los ya procesados
    audio_groups = list(df_win.groupby("path"))

    # ---> REANUDAR (por poca gpu se cortaba) <---
    existing_shards = sorted(glob.glob(str(out_dir / "shard_*.npz")))
    shard_id = 0
    files_to_skip = 0
    if existing_shards:
        last_shard_file = existing_shards[-1]
        last_shard_num = int(re.search(r'shard_(\d+).npz', str(last_shard_file)).group(1))
        shard_id = last_shard_num + 1
        files_to_skip = shard_id * FILES_PER_SHARD
        print(f"Se encontraron {len(existing_shards)} lotes existentes. Reanudando desde el lote {shard_id}.")
        print(f"Se saltarán los primeros {files_to_skip} archivos de audio.")

    shard_features_list = []
    files_in_shard = 0

    # Usamos un iterador para poder saltar los archivos ya procesados
    pbar = tqdm(audio_groups[files_to_skip:], total=len(audio_groups)-files_to_skip, desc=f"Shard {shard_id}")

    for path_audio, group in pbar:
        try:
            pbar.set_postfix_str(f"Procesando: {Path(path_audio).name}")
            y_cpu, sr = librosa.load(str(path_audio), sr=SR, mono=True)
            rms = np.sqrt(np.mean(y_cpu**2))
            if rms < MIN_RMS_THRESHOLD:
                for _ in range(len(group)):
                    shard_features_list.append(np.zeros(30, dtype=np.float32))
                continue

            y_gpu = torch.from_numpy(y_cpu).to(device)
            with torch.no_grad():
                full_mfccs = mfcc_transform(y_gpu)
                full_pitch, full_periodicity = torchcrepe.predict(y_gpu.unsqueeze(0), sr, device=device, return_periodicity=True)

            for _, row in group.iterrows():
                start_frame_mfcc = row["start_samp"] // HOP_LENGTH_MFCC
                end_frame_mfcc = row["end_samp"] // HOP_LENGTH_MFCC
                mfcc_segment = full_mfccs[:, start_frame_mfcc:end_frame_mfcc]
                if mfcc_segment.shape[1] > 0:
                    mfcc_mean, mfcc_std = mfcc_segment.mean(axis=1), mfcc_segment.std(axis=1)
                else:
                    mfcc_mean, mfcc_std = torch.zeros(13, device=device), torch.zeros(13, device=device)

                start_frame_pitch = row["start_samp"] // 160
                end_frame_pitch = row["end_samp"] // 160
                pitch_segment = full_pitch[0, start_frame_pitch:end_frame_pitch]
                periodicity_segment = full_periodicity[0, start_frame_pitch:end_frame_pitch]
                pitch_voiced = pitch_segment[periodicity_segment > 0.1]

                if pitch_voiced.shape[0] > 1:
                    pitch_features = torch.stack([pitch_voiced.mean(), pitch_voiced.std(), pitch_voiced.min(), pitch_voiced.max()])
                elif pitch_voiced.shape[0] == 1:
                    val = pitch_voiced[0]
                    pitch_features = torch.stack([val, torch.tensor(0.0, device=device), val, val])
                else:
                    pitch_features = torch.zeros(4, device=device)

                all_feats_gpu = torch.cat([mfcc_mean, mfcc_std, pitch_features])
                shard_features_list.append(all_feats_gpu.cpu().numpy())

        except Exception as e:
            print(f"\n[ERROR] en {Path(path_audio).name}: {e}. Rellenando con ceros.")
            for _ in range(len(group)):
                shard_features_list.append(np.zeros(30, dtype=np.float32))

        files_in_shard += 1

        if files_in_shard >= FILES_PER_SHARD:
            print(f"\nGuardando lote {shard_id}...")
            features_matrix = np.stack(shard_features_list, axis=0).astype(np.float32)
            out_path = out_dir / f"shard_{shard_id:03d}.npz"
            np.savez_compressed(out_path, acoustic_features=features_matrix)
            print(f"Lote {shard_id} guardado en {out_path} con shape {features_matrix.shape}")

            shard_features_list = []
            files_in_shard = 0
            shard_id += 1
            pbar.set_description(f"Shard {shard_id}")

    if shard_features_list:
        print(f"\nGuardando el último lote {shard_id}...")
        features_matrix = np.stack(shard_features_list, axis=0).astype(np.float32)
        out_path = out_dir / f"shard_{shard_id:03d}.npz"
        np.savez_compressed(out_path, acoustic_features=features_matrix)
        print(f"Último lote guardado con shape {features_matrix.shape}")


In [ ]:
extract_features(TRAIN_WIN_NPZ, SHARD_OUTPUT_DIR)

Usando dispositivo: cuda
Se encontraron 15 lotes existentes. Reanudando desde el lote 15.
Se saltarán los primeros 15000 archivos de audio.


Shard 15:   0%|          | 0/5151 [00:00<?, ?it/s]


Guardando lote 15...
Lote 15 guardado en /content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features_shards_train/shard_015.npz con shape (7429, 30)

Guardando lote 16...
Lote 16 guardado en /content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features_shards_train/shard_016.npz con shape (6479, 30)

Guardando lote 17...
Lote 17 guardado en /content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features_shards_train/shard_017.npz con shape (6461, 30)

Guardando lote 18...
Lote 18 guardado en /content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features_shards_train/shard_018.npz con shape (6563, 30)

Guardando lote 19...
Lote 19 guardado en /content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features_shards_train/shard_019.npz con shape (7524, 30)

Guardando el último lote 20...
Último lote guardado con shape (1114, 30)


In [ ]:
extract_features(
    TEST_WIN_NPZ,
    ACOUSTIC_FEAT_DIR / "test_acoustic_features_v3.npz"
)

Usando dispositivo: cuda


Shard 0:   0%|          | 0/2428 [00:00<?, ?it/s]


Guardando lote 0...
Lote 0 guardado en /content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features/test_acoustic_features_v3.npz/shard_000.npz con shape (6716, 30)

Guardando lote 1...
Lote 1 guardado en /content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features/test_acoustic_features_v3.npz/shard_001.npz con shape (6251, 30)

Guardando el último lote 2...
Último lote guardado con shape (2765, 30)


In [ ]:
import numpy as np
from pathlib import Path
import glob

def merge_shards(shard_dir: Path, final_out_path: Path):
    """
    Encuentra todos los archivos de lotes, los carga y los une en un solo archivo.
    """
    shard_files = sorted(glob.glob(str(shard_dir / "shard_*.npz")))

    if not shard_files:
        print("No se encontraron archivos de lotes (shards) en el directorio.")
        return

    print(f"Encontrados {len(shard_files)} lotes. Uniéndolos...")

    all_features = []
    for shard_file in shard_files:
        with np.load(shard_file) as data:
            all_features.append(data['acoustic_features'])

    final_matrix = np.concatenate(all_features, axis=0)

    np.savez_compressed(final_out_path, acoustic_features=final_matrix)
    print(f"¡Listo! Archivo final guardado en {final_out_path}")
    print(f"Shape de la matriz final: {final_matrix.shape}")


In [ ]:
# Directorio donde se guardaron los lotes
SHARD_OUTPUT_DIR = Path("/content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features_shards_train")

FINAL_NPZ_PATH = Path("/content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features/train_acoustic_features_v3.npz")

merge_shards(SHARD_OUTPUT_DIR, FINAL_NPZ_PATH)

Encontrados 21 lotes. Uniéndolos...
¡Listo! Archivo final guardado en /content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features/train_acoustic_features_v3.npz
Shape de la matriz final: (132621, 30)


In [ ]:
# Directorio donde se guardaron los lotes
SHARD_OUTPUT_DIR = Path("/content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features/test_acoustic_features_v3")

FINAL_NPZ_PATH = Path("/content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features/test_acoustic_features_v3.npz")

merge_shards(SHARD_OUTPUT_DIR, FINAL_NPZ_PATH)

Encontrados 3 lotes. Uniéndolos...
¡Listo! Archivo final guardado en /content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features/test_acoustic_features_v3.npz
Shape de la matriz final: (15732, 30)


# wav2vec

In [ ]:
BASE_V3   = Path("/content/drive/MyDrive/TP1/data/deriv/v3")
WIN_DIR   = BASE_V3 / "windows"
OUT_DIR   = BASE_V3 / "embeddings"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_WIN = WIN_DIR / "windows_train_v3.npz"
TEST_WIN  = WIN_DIR / "windows_test_v3.npz"

In [ ]:
SR       = 16000
MODEL_ID = "facebook/wav2vec2-large-xlsr-53"
POOLING  = "meanstd"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)

fe  = AutoFeatureExtractor.from_pretrained(MODEL_ID)
w2v = Wav2Vec2Model.from_pretrained(MODEL_ID).to(DEVICE).eval()

# stride de frames del encoder (320 para wav2vec2)
FRAME_STRIDE = int(np.prod(getattr(w2v.config, "conv_stride", [320])))
# D_POOL = 2*HID # esto ya lo hace lo omitimos

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

## funciones

In [ ]:
def load_windows_v3(npz_path: Path) -> pd.DataFrame:
    D = np.load(npz_path, allow_pickle=True)
    path  = D["path"].astype("U")
    start = D["start_samp"].astype(np.int64)
    end   = D["end_samp"].astype(np.int64)
    zrms  = D["zrms"].astype(np.float32)
    lang  = D["lang"].astype("U")
    # (opcional) clip_idx = D["clip_idx"].astype(np.int64)

    df = pd.DataFrame({
        "path": path,
        "start_samp": start,
        "end_samp": end,
        "zrms": zrms,
        "lang": lang,
    })
    # ordenar para que los índices de slicing sean estables
    df = df.sort_values(["path", "start_samp"]).reset_index(drop=True)
    return df

In [ ]:
def safe_load_audio(path, sr=SR):
    y, _ = librosa.load(path, sr=sr, mono=True)
    y = y.astype(np.float32)
    return y - y.mean()

In [ ]:
@torch.no_grad()
def forward_hidden(y: np.ndarray) -> np.ndarray:
    inputs = fe([y], sampling_rate=SR, return_tensors="pt", padding=True)
    iv = inputs["input_values"].to(DEVICE)
    am = inputs.get("attention_mask")
    if am is not None: am = am.to(DEVICE)
    H = w2v(input_values=iv, attention_mask=am).last_hidden_state.squeeze(0)  # [T', D]
    H_np = H.detach().cpu().numpy().astype(np.float32)
    del inputs, iv, am, H
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return H_np

In [ ]:
def pool_windows_meanstd_from_frames(H_np: np.ndarray, starts: np.ndarray, ends: np.ndarray) -> np.ndarray:
    Tprime, D = H_np.shape
    s_f = np.floor(starts / FRAME_STRIDE).astype(np.int64)
    e_f = np.ceil(  ends  / FRAME_STRIDE).astype(np.int64)
    s_f = np.clip(s_f, 0, max(Tprime-1, 0))
    e_f = np.clip(e_f, 1, Tprime)

    cs  = np.cumsum(H_np, axis=0)
    cs2 = np.cumsum(H_np*H_np, axis=0)

    if POOLING == "mean":
        X = np.empty((len(s_f), D), dtype=np.float32)
    else:  # meanstd
        X = np.empty((len(s_f), 2*D), dtype=np.float32)

    for i, (s, e) in enumerate(zip(s_f, e_f)):
        if e <= s:
            e = min(s+1, Tprime)
        L    = float(e - s)
        sum_ = cs[e-1] - (cs[s-1] if s>0 else 0.0)
        mean = sum_ / L
        if POOLING == "mean":
            X[i, :] = mean.astype(np.float32)
        else:
            sum2 = cs2[e-1] - (cs2[s-1] if s>0 else 0.0)
            var  = np.maximum(sum2 / L - mean*mean, 1e-12)
            std  = np.sqrt(var, dtype=np.float32)
            X[i, :D] = mean.astype(np.float32)
            X[i, D:] = std
    return X

In [ ]:
def extract_embeddings_from_windows(npz_in: Path):
    print(f"\n== Embeddings desde {npz_in.name} ==")
    dfw = load_windows_v3(npz_in)

    X_list, zrms_list, lang_list, path_list = [], [], [], []

    # 1 forward por audio completo:
    for wav, g in tqdm(dfw.groupby("path", sort=False), total=dfw["path"].nunique()):
        y = safe_load_audio(wav, sr=SR)
        H = forward_hidden(y)  # [T', D]
        starts = g["start_samp"].to_numpy(np.int64)
        ends   = g["end_samp"].to_numpy(np.int64)

        Xw = pool_windows_meanstd_from_frames(H, starts, ends)  # [n_g, D or 2D]

        X_list.append(Xw)
        zrms_list.append(g["zrms"].to_numpy(np.float32))
        lang_list.append(g["lang"].to_numpy("U"))
        path_list.append(g["path"].to_numpy("U"))

        del y, H, Xw
        gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache()
    return X_list, zrms_list, lang_list, path_list

In [ ]:
def extract_embeddings_from_windows_sharded(
    npz_in: Path,
    out_prefix: Path,
    shard_nwins: int = 12000,
    resume: bool = True
):
    """
    Lee ventanas v3 (path, start_samp, end_samp, zrms, lang) y escribe shards .npz
    con {Xw, path, zrms, lang}. Cada shard tiene ~shard_nwins ventanas.
    Crea:
      - <out_prefix>_shards/XXXX.npz
      - <out_prefix>_shards/index.json
      - <out_prefix>_shards/done_paths.json   (para reanudar)
    """
    shards_dir = out_prefix.parent / f"{out_prefix.stem}_shards"
    shards_dir.mkdir(parents=True, exist_ok=True)
    index_json = shards_dir / "index.json"
    done_json  = shards_dir / "done_paths.json"

    print(f"\n== Embeddings desde {npz_in.name} ==")
    dfw = load_windows_v3(npz_in)  # columns: path, start_samp, end_samp, zrms, lang

    # Reanudar: cargar paths ya hechos
    done_paths = set()
    shards_idx = {"meta": {}, "shards": []}
    if resume:
        if done_json.exists():
            try:
                done_paths = set(json.loads(done_json.read_text()))
            except Exception:
                done_paths = set()
        if index_json.exists():
            try:
                shards_idx = json.loads(index_json.read_text())
            except Exception:
                shards_idx = {"meta": {}, "shards": []}

    # Buffers para el shard
    X_buf, zrms_buf, lang_buf, path_buf = [], [], [], []
    total_written = sum(s.get("nrows", 0) for s in shards_idx.get("shards", []))
    shard_id = len(shards_idx.get("shards", []))

    # util para volcar un shard
    def _flush():
        nonlocal X_buf, zrms_buf, lang_buf, path_buf, shard_id, total_written, shards_idx

        if not X_buf:
            return
        Xcat   = np.concatenate(X_buf, axis=0).astype(np.float32)
        z_cat  = np.concatenate(zrms_buf, axis=0).astype(np.float32)
        l_cat  = np.concatenate(lang_buf, axis=0).astype("U")
        p_cat  = np.concatenate(path_buf, axis=0).astype("U")

        shard_path = shards_dir / f"shard_{shard_id:05d}.npz"
        np.savez_compressed(
            shard_path,
            Xw=Xcat,
            zrms=z_cat,
            lang=l_cat,
            path=p_cat
        )
        shards_idx["shards"].append({
            "path": str(shard_path),
            "nrows": int(Xcat.shape[0]),
        })
        shard_id += 1
        total_written += int(Xcat.shape[0])

        # limpiar buffers
        X_buf, zrms_buf, lang_buf, path_buf = [], [], [], []

        # persistir índices y done_paths para reanudar
        shards_idx["meta"] = {
            "npz_source": str(npz_in),
            "sr": int(SR),
            "pooling": "meanstd",
            "device": DEVICE,
            "total_written": int(total_written),
        }
        index_json.write_text(json.dumps(shards_idx, indent=2))
        done_json.write_text(json.dumps(sorted(list(done_paths))))

        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    # loop por audio (1 forward por wav)
    gb = dfw.groupby("path", sort=False)
    pbar = tqdm(gb, total=dfw["path"].nunique(), desc="Audios")
    for wav, g in pbar:
        if resume and wav in done_paths:
            continue

        # carga y forward
        y = safe_load_audio(wav, sr=SR)
        H = forward_hidden(y)  # [T', D]
        starts = g["start_samp"].to_numpy(np.int64)
        ends   = g["end_samp"].to_numpy(np.int64)

        # pooling por ventana (usa SR global; fallback si tu firma no pide SR)
        try:
            Xw = pool_windows_meanstd_from_frames(H, starts, ends, SR)
        except TypeError:
            Xw = pool_windows_meanstd_from_frames(H, starts, ends)

        X_buf.append(Xw)
        zrms_buf.append(g["zrms"].to_numpy(np.float32))
        lang_buf.append(g["lang"].to_numpy("U"))
        path_buf.append(g["path"].to_numpy("U"))

        done_paths.add(wav)

        # ¿toca volcar?
        n_buf = sum(x.shape[0] for x in X_buf)
        if n_buf >= shard_nwins:
            _flush()
            pbar.set_postfix_str(f"guardado={total_written}")

        # limpia por archivo
        del y, H, Xw
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    # shard final
    _flush()

    print(f"[OK] Shards en {shards_dir} | total ventanas={total_written}")
    return shards_dir  # por si lo querés usar enseguida


def merge_shards_to_npz(index_json: Path, out_npz: Path):
    """Une los shards creados arriba en un solo .npz con {Xw, path, zrms, lang}."""
    idx = json.loads(Path(index_json).read_text())
    shards = idx.get("shards", [])
    if not shards:
        raise RuntimeError("No hay shards en el índice.")

    Xs, Zs, Ls, Ps = [], [], [], []
    for s in shards:
        D = np.load(s["path"])
        Xs.append(D["Xw"])
        Zs.append(D["zrms"])
        Ls.append(D["lang"])
        Ps.append(D["path"])

    Xw   = np.concatenate(Xs, axis=0).astype(np.float32)
    zrms = np.concatenate(Zs, axis=0).astype(np.float32)
    lang = np.concatenate(Ls, axis=0).astype("U")
    path = np.concatenate(Ps, axis=0).astype("U")

    np.savez_compressed(out_npz, Xw=Xw, path=path, zrms=zrms, lang=lang)
    print(f"[OK] Final guardado en: {out_npz} | X shape: {Xw.shape}")


## resultados

In [ ]:
npz_out = OUT_DIR / "test_embeddings_v3.npz"

In [ ]:
X_list, zrms_list, lang_list, path_list  = extract_embeddings_from_windows(TEST_WIN)


== Embeddings desde windows_test_v3.npz ==


  0%|          | 0/2428 [00:00<?, ?it/s]

In [ ]:
Xw   = np.concatenate(X_list, axis=0).astype(np.float32)
zrms = np.concatenate(zrms_list, axis=0).astype(np.float32)
lang = np.concatenate(lang_list, axis=0).astype("U")
path = np.concatenate(path_list, axis=0).astype("U")

np.savez_compressed(
    npz_out,
    Xw=Xw,
    path=path,
    zrms=zrms,
    lang=lang,
    sr=np.int32(SR),
    pooling=np.bytes_(POOLING.encode("utf-8")),
)
print("Guardado:", npz_out, "| Xw shape:", Xw.shape)

Guardado: /content/drive/MyDrive/TP1/data/deriv/v3/embeddings/test_embeddings_v3.npz | Xw shape: (15732, 2048)


In [ ]:
train_shards = extract_embeddings_from_windows_sharded(
    Path("/content/drive/MyDrive/TP1/data/deriv/v3/windows/windows_train_v3.npz"),
    Path("/content/drive/MyDrive/TP1/data/deriv/embeddings/train_v3"),
    shard_nwins=12000,
    resume=True
)


== Embeddings desde windows_train_v3.npz ==


Audios:   0%|          | 0/20151 [00:00<?, ?it/s]

[OK] Shards en /content/drive/MyDrive/TP1/data/deriv/embeddings/train_v3_shards | total ventanas=132621


In [ ]:
merge_shards_to_npz(train_shards / "index.json",
                    Path("/content/drive/MyDrive/TP1/data/deriv/embeddings/train_embeddings_v3.npz"))


[OK] Final guardado en: /content/drive/MyDrive/TP1/data/deriv/embeddings/train_embeddings_v3.npz | X shape: (132621, 2048)
